# Day 07 · 文件 I/O 与异常处理

> **前置**: Day01-06 已掌握变量、字符串、分支、循环、函数、列表字典
> **关键问题**: 程序一关数据就丢,怎么持久化到磁盘?程序动不动就崩溃,怎么穿着防弹衣跑?

**引入:日记本为何能翻旧页**

抽问上节: 列表和字典分别是"什么"?(一排长格子 / 带 key 的快递柜)。再问: 在程序里定义的变量,关掉窗口就没了 —— 用过日记本吗?第一次写的第二天还能翻到,这就是**持久化**:程序 ↔ 硬盘之间的桥梁,学名**文件 I/O**。


**1. `with open()` 文件读写**

类比: 打开日记本(`open`) → 写/读(`write`/`read`) → 合上本子。Python 推荐**`with` 语句**,出块自动 `close()`,即便中间报错也不会漏关 —— 永不漏写。三种基础模式:`'r'` 读(默认)、`'w'` 写(覆盖)、`'a'` 追加。


In [ ]:
# 写文件(覆盖模式)
with open("diary.txt", "w", encoding="utf-8") as f:
    f.write("2026-07-08 晴\n")
    f.write("今天学了文件 I/O\n")
# 出 with 块,f 已自动关闭

# 读文件(默认 'r')
with open("diary.txt", "r", encoding="utf-8") as f:
    content = f.read()          # 一次性读完全部
print(content)


三种读法:`read()` 整个文件 → 一个字符串;`readline()` 只读一行;`readlines()` 全部读入 → 每行作为字符串的列表。


In [ ]:
lines = ["语文", "数学", "英语", "物理"]
# 用 writelines(不会自动加换行,手动拼)
with open("course.txt", "w", encoding="utf-8") as f:
    f.writelines(line + "\n" for line in lines)

# 逐行读(readlines)
with open("course.txt", "r", encoding="utf-8") as f:
    rows = f.readlines()        # ['语文\n', '数学\n', ...]
print("readlines:", rows)


**2. `json` 读写结构化数据**

纯文本 `"苹果,香蕉"` 无法表达列表/字典嵌套。**JSON** 是程序之间交换数据的"通用语言",Python 内置 `json` 模块。口诀:**`dump` 写入文件,`load` 读出文件;末尾的 `s` = string(字符串)**。


In [ ]:
import json

data = {
    "name": "小明",
    "scores": [90, 85, 92],
    "is_vip": True
}

# 写入 JSON 文件(中文不转 \uXXXX)
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# 读取 JSON 文件
with open("data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded["scores"])         # [90, 85, 92]

# 字符串 ↔ 字典
s = json.dumps(data, ensure_ascii=False)
print(json.loads(s)["name"])    # 小明


**3. `try` / `except` / `else` / `finally` 异常处理**

程序遇到错误会**崩溃退出**。`try-except` 让我们"抓住"错误决定下一步,而不是直接死掉。口诀:**`try` 放危险代码,`except` 按类型分流,`else` 无错才跑,`finally` 必定执行**(常用来关文件/清理资源)。


In [ ]:
try:
    num = int(input("请输入整数:"))
    result = 100 / num
except ValueError:
    print("输入的不是整数!")
except ZeroDivisionError:
    print("不能除以 0!")
else:
    print("计算结果是:", result)    # 没异常才跑
finally:
    print("无论有无异常都执行")


常见异常速查:`FileNotFoundError`(文件不存在)、`ValueError`(`int("abc")` 转换失败)、`TypeError`(`"hi" + 1`)、`KeyError`(访问字典不存在的 key)、`IndexError`(列表越界)。永远**指定具体异常类型**,别裸 `except:`,那会吞掉真正的 Bug。


In [ ]:
# 用异常处理读 JSON 文件,崩不了
import json, os

FILE = "diary.json"

def load():
    if not os.path.exists(FILE):
        return []                   # 文件不存在 → 空列表
    with open(FILE, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []               # 文件内容非法 → 空列表

print("load 结果:", load())


**今日小结**

三大块:`with open` 安全读写文件、`json.load/dump` 持久化结构化数据、`try-except-else-finally` 异常防护。

**练习 1 ⭐⭐**:写 3 行日记到 `diary.txt`,再 `readlines()` 逐行打印。

**练习 2 ⭐⭐**:把通讯录字典用 `json.dump` 写入文件,再 `json.load` 读回,改动一项比较前后。

**练习 3 ⭐⭐⭐**:给除法器加异常防护,输错 3 次弹 `ValueError` 友好提示后退出。

> 🔴 **易错点**:漏写 `encoding="utf-8"`,中文路径直接报 `UnicodeDecodeError`;裸 `except:` 吞掉所有异常包括 `Ctrl+C`。
